# 02 Prompting
In this workbook we'll cover various prompting techniques with Agents & models.

In [1]:
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [2]:
# create an Agent
from langchain.agents import create_agent

agent = create_agent(model="openai:gpt-5-nano")

Following is the simplest way of calling an agent using `HumanMessage`

In [3]:
from langchain.messages import HumanMessage

question = HumanMessage(content="What's the capital of the moon?")

response = agent.invoke({"messages": [question]})
print(response["messages"][1].content)

There isn’t one. The Moon isn’t a country or a city, and it has no government or permanent inhabitants, so it has no capital. If you’re thinking fiction, some stories invent capitals or bases like “Luna City” or other names, but nothing real exists. Would you like a fun fictional suggestion?


### Adding System Prompts

You can always add system prompts to an agent, wjoh act as the foundational, high-level instructions that define its persona, behavior, constraints, and operational goals before any user interaction.

This section shows you how to add system prompts.

In [4]:
system_prompt = """You are a science fiction writer, that generates innovative 
    responses to user's requests"""

agent = create_agent(
    model="openai:gpt-5-nano",
    # define system prompt like this
    system_prompt=system_prompt,
)

In [5]:
# and now ask the same question
question = HumanMessage(content="What's the capital of the moon?")

response = agent.invoke({"messages": [question]})
print(response["messages"][1].content)

In reality there isn’t one—the Moon isn’t a country—so there’s no official capital. If you’re building a science-fiction world, here are a few tasty options you might use:

- Selene City (near side)
  - A glittering glass-domed capital perched along the terminator, home to the Lunar Assembly and the Federation’s central data hubs. Neon-reflective seas and a splashy, high-tech feel; daily life is a blend of bureaucratic ceremony and improvisational moon-dweeb engineering.

- Lunaris (Lunar Federation capital, in a crater)
  - Built inside a sheltered crater (think Shackleton or a Mare Imbrium rim). Subterranean and surface districts connected by spiral transit rails, lit by bioluminescent flora. Governing council sits in a caverns-turned-congress hall; the city runs on ice-water, solar energy, and rumor.

- Artemis Prime (the terminator hub)
  - A capital perched on the edge where day meets night, with a colossal spaceport and orbital elevator. A techno-corporate republic with a heady m

You can also add few-shot examples to your system prompt to make the agent generate better responses.

In [6]:
system_prompt = """

You are a science fiction writer, create a space capital city at the users request.

User: What is the capital of mars?
Scifi Writer: Marsialis

User: What is the capital of Venus?
Scifi Writer: Venusovia

"""

agent = create_agent(model="gpt-5-nano", system_prompt=system_prompt)

question = HumanMessage(content="What's the capital of the moon?")

response = agent.invoke({"messages": [question]})

print(response["messages"][1].content)

Scifi Writer: Lunaria


### Structured Prompts

Here we show you how structured system prompts can control how models generate output - this is _different_ from structured outputs, which we will cover later.

In [7]:
system_prompt = """

You are a science fiction writer, create a space capital city at the users request.
Please keep to the below structure.

Name: The name of the capital city
Location: Where it is based
Vibe: 2-3 words to describe its vibe
Economy: Main industries
"""

agent = create_agent(model="gpt-5-nano", system_prompt=system_prompt)

question = HumanMessage(content="What's the capital of the moon?")

response = agent.invoke({"messages": [question]})

print(response["messages"][1].content)

Name: Selene Prime
Location: Near-side equatorial lunar lava-tube megacity beneath Mare Serenitatis; solar domes and relay links connect to Earth.
Vibe: Futuristic frontier
Economy: Helium-3 mining; water/ice extraction; solar energy production; lunar construction and spaceport services; scientific research


### Structured Output

Models generate text, which is unstructured format. Agents are often deployed in teams, with one agent handing off control to another. In such a situation, it becomes critical to ensure that _all pieces_ of data required are extracted from outputs generated by models in a specific structure.

In [8]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from pydantic import BaseModel


class CapitalInfo(BaseModel):
    name: str
    location: str
    vibe: str
    economy: str


agent = create_agent(
    model="gpt-5-nano",
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
    response_format=CapitalInfo,
)

question = HumanMessage(content="What is the capital of The Moon?")

response = agent.invoke({"messages": [question]})

response["structured_response"]

CapitalInfo(name='Selene Prime', location='Rim of Shackleton Crater, lunar south pole (permanently shadowed region)', vibe='A glittering, glass-domed metropolis of anti-grav transit, neon-lit avenues, and lush hydroponic parks.', economy='Helium-3 and water-ice mining, lunar construction, in-situ manufacturing, bioscience, and orbital trade.')

In [10]:
# you can access individual fields like this
response["structured_response"].name, response["structured_response"].location

('Selene Prime',
 'Rim of Shackleton Crater, lunar south pole (permanently shadowed region)')